# Explainable AI (XAI) Framework for Psychometric Assessment
## Empirical Research Foundation on DASS-42

This research notebook presents a comprehensive **Explainable AI Framework** for psychometric assessment and mental health screening built upon empirical responses from the **Depression Anxiety Stress Scales (DASS-42)** ($N = 34,576$ validated participants).

### Core Project Objectives Achieved:
1. **Task 1: End-to-End XAI Framework**: Full integration of machine learning and neural architectures with state-of-the-art explainability methods.
2. **Task 2: 6 Explainability Techniques & Comparative Evaluation**:
   - **SHAP** (Shapley Additive exPlanations)
   - **LIME** (Local Interpretable Model-agnostic Explanations)
   - **Integrated Gradients** (Axiomatic path integrals)
   - **Grad-CAM** (1D Convolutional class activation mapping)
   - **Attention-Based Explanations** (Transformer multi-head self-attention)
   - **Counterfactual Recourse** (Actionable clinical intervention search)
   - **Quantitative Comparison Benchmark** (Faithfulness, Gini Sparsity, Concordance)
3. **Task 3: Multimodal Psychometric Fusion & Explainability**:
   - Synchronized fusion of 5 clinical streams: Survey Responses, Behavioural Timings, Clinical Text Narratives, Speech Acoustics, and Physiological Signals.

### 1. Environment Setup and Data Loading
We begin by importing required packages and loading the official Open Source Psychometrics DASS-42 dataset.

In [ ]:
import os
import sys
sys.path.insert(0, os.path.abspath('..'))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from src.data.loader import download_and_load_data, inspect_dataset
from src.data.preprocessing import preprocess_data

# Load and clean empirical research dataset
df_clean = download_and_load_data(raw_dir='../data/raw')
stats = inspect_dataset(df_clean)

### 2. Psychometric Preprocessing and Stratified Split
Map raw survey items to human-readable psychological labels and partition into stratified train/test sets.

In [ ]:
X_train, X_test, y_train, y_test, feature_names = preprocess_data(
    df_clean, test_size=0.2, random_state=42
)
print(f"Training instances: {len(X_train):,} | Test instances: {len(X_test):,}")
print(f"Predictor feature dimensions: {len(feature_names)}")

### 3. Model Training & Evaluation
Train baseline classifier and evaluate classification metrics (ROC-AUC, F1, Confusion Matrix).

In [ ]:
from src.models.baseline_model import train_baseline_model
from src.evaluation.metrics import evaluate_model

model = train_baseline_model(X_train, y_train, model_type='random_forest', random_state=42)
eval_metrics = evaluate_model(model, X_test, y_test, output_dir='../outputs/figures')

### 4. Explainability Techniques (Task 1 & Task 2)
We now execute each of the 6 explainability techniques for a representative participant.

In [ ]:
from src.explainers.shap_explainer import generate_shap_explanations
from src.explainers.lime_explainer import generate_lime_explanations
from src.explainers.integrated_gradients import generate_integrated_gradients_explanations
from src.explainers.grad_cam import generate_gradcam_explanations
from src.explainers.attention_explainer import generate_attention_explanations
from src.explainers.counterfactual import generate_counterfactual_explanations

sample_idx = 0

# 1. SHAP
shap_res = generate_shap_explanations(model, X_train, X_test, feature_names, output_dir='../outputs/shap', sample_idx=sample_idx)

# 2. LIME
lime_res = generate_lime_explanations(model, X_train, X_test, feature_names, output_dir='../outputs/lime', sample_idx=sample_idx)

# 3. Integrated Gradients
ig_res = generate_integrated_gradients_explanations(model, X_train, X_test, feature_names, output_dir='../outputs/ig', sample_idx=sample_idx)

# 4. Grad-CAM
cam_res = generate_gradcam_explanations(model, X_train, X_test, feature_names, output_dir='../outputs/gradcam', sample_idx=sample_idx)

# 5. Attention Explanations
attn_res = generate_attention_explanations(model, X_train, X_test, feature_names, output_dir='../outputs/attention', sample_idx=sample_idx)

# 6. Counterfactual Recourse
cf_res = generate_counterfactual_explanations(model, X_train, X_test, feature_names, output_dir='../outputs/counterfactual', sample_idx=sample_idx)

### 5. Task 3: Multimodal Psychometric Fusion & Explainability
Synchronize 5 modalities (Questionnaire, Timing Latency, Clinical Narrative Text, Speech Acoustics, and Autonomic Physiology) and train the attention-gated multimodal fusion network.

In [ ]:
from src.multimodal.architecture import MultimodalXAIArchitecture

multimodal_arch = MultimodalXAIArchitecture()
mm_results = multimodal_arch.run_pipeline(df_clean, sample_idx=sample_idx, output_dir='../outputs/multimodal')
print('Multimodal Predicted Risk:', mm_results['risk_prob'])
print('Modality Attention Weights:', mm_results['modality_weights_pct'])

### 6. Task 2: Quantitative Comparative Evaluation Across XAI Techniques
Benchmark Faithfulness (Area Under Deletion Curve), Gini Sparsity, Concordance (Spearman Rank Correlation), and Runtime.

In [ ]:
from src.evaluation.xai_comparison import run_xai_comparison_benchmark

benchmark_results = run_xai_comparison_benchmark(
    model, X_train, X_test, feature_names, output_dir='../outputs/comparison', sample_idx=sample_idx
)

### 7. Interactive Research Dashboard Generation
Finally, compile all calculated metrics, explainability plots, and interactive simulation widgets into a standalone HTML5 portal.

In [ ]:
from src.dashboard.dashboard_generator import generate_interactive_dashboard

dashboard_file = generate_interactive_dashboard(
    metrics=eval_metrics, sample_idx=sample_idx, output_path='../outputs/dashboard/index.html'
)
print(f'Interactive Portal generated at: {dashboard_file}')